# Static Field Simulation & Visualization
## Multipath Passive Radar Simulation — Visualizer

This notebook provides an end-to-end workflow to:

- Define a simulation scene
- Generate static multipath ray fields
- Cache and reuse results
- Visualize propagation paths in 3D

> **TODO**  
> Integrate UAV support into this workflow.

---

## Quick Start

1. Define the simulation domain and engine parameters (Cell 1)  
2. Define geometry, transmitters, and receivers; then run static field generation (Cell 2)  
3. Select a cached static field (Cell 3)  
4. Visualize the rays of the selected field (Cell 4)

In [1]:
#@title ── 0. Environment Setup { display-mode: "form" }
# Clones the repository (skips if already present), installs the package,
# and verifies GPU availability. Run this before every other cell.
import os, sys, subprocess

REPO_NAME = "multipath-passive-radar-sim"
REPO_PATH = f"/content/{REPO_NAME}"

if not os.path.exists(REPO_PATH):
    print("Cloning repository...")
    subprocess.run(
        ["git","clone",f"https://github.com/dealmeidapaulo/{REPO_NAME}.git",REPO_PATH],
        capture_output=True,
    )

%cd {REPO_PATH}

SRC_PATH = os.path.join(REPO_PATH, "src")
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

subprocess.run(["pip","install","-e",".","-q"], capture_output=True)

import numpy as np
from src.core.scene.domain import Scene, Box, Obstacle, Transmitter, Receiver, UAV
from src.core.cache import get_or_compute

print("Imports OK")

r = subprocess.run(
    ["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
    capture_output=True, text=True,
)
print(f"GPU: {r.stdout.strip() if r.returncode==0 else 'None -- change runtime to T4'}")


Cloning repository...
/content/multipath-passive-radar-sim
Imports OK
GPU: Tesla T4, 15360 MiB


# Static Field Generation

Define the simulation domain and compute the **static ray field**: the complete set of multipath propagation paths, traced independently of any UAV or dynamic target.

This step is typically the most computationally expensive, and its results are **cached** for reuse.

---

## Domain
Defines the 3D simulation bounds (in meters). All geometry, transmitters, and receivers must lie within this region.

## Ray Tracing
- **Ray count** controls sampling density (higher → more accurate, slower)
- **Max bounces** limits the number of reflections per path
- **Roughness** controls surface scattering (0 = specular, 1 = diffuse)

## Spatial Acceleration
- **Spatial hash cell size** controls spatial grouping of rays  
- Typically chosen on the order of the expected interaction scale (e.g., UAV detection radius)

## Caching
Generated fields are cached using a hash of:
- Domain configuration
- Geometry and entities
- Ray-tracing parameters

If a matching configuration exists, it is loaded instead of recomputed.

In [2]:
#@title ── 1. Domain & Engine Parameters { display-mode: "form" }
import numpy as np

# -- Domain extent ------------------------------------------------------------
DOMAIN_X = 100   #@param {type:"number"}
DOMAIN_Y = 100   #@param {type:"number"}
DOMAIN_Z = 100   #@param {type:"number"}

DOMAIN_SIZE = [DOMAIN_X, DOMAIN_Y, DOMAIN_Z]

#@markdown ---
#@markdown ### Ray-tracer settings

# Maximum reflections per ray before the path is terminated.
MAX_BOUNCES = 4   #@param {type:"slider", min:1, max:10, step:1}

# Total rays emitted per TX.
# 100k ~ 1 s on T4  |  500k ~ 5 s  |  750k ~ 8 s
RAY_COUNT = 30000   #@param [30000, 60000, 100000, 250000, 500000, 750000] {type:"raw"}

# Spatial hash cell size (m). Rule of thumb: equal to UAV detection radius.
SPATIAL_HASH_CELL = 25.0   #@param {type:"number"}

# Surface roughness: 0 = perfect mirror, 1 = fully diffuse (Lambertian).
ROUGHNESS = 0.6   #@param {type:"slider", min:0.0, max:1.0, step:0.05}

print(f"Domain   : {DOMAIN_X} x {DOMAIN_Y} x {DOMAIN_Z} m")
print(f"Rays     : {RAY_COUNT:,}  |  Max bounces: {MAX_BOUNCES}  |  Roughness: {ROUGHNESS}")
print(f"Hash cell: {SPATIAL_HASH_CELL} m")


Domain   : 100 x 100 x 100 m
Rays     : 30,000  |  Max bounces: 4  |  Roughness: 0.6
Hash cell: 25.0 m


In [3]:
#@title ── 2. Interactive Scene Editor & Field Precompute { display-mode: "form" }
# Drag TX / RX / buildings on the 2-D canvas and inspect them in 3-D.
# Click any element to edit its properties.
# Press SYNCHRONIZE & COMPUTE FIELD to launch the GPU precompute.
#
# The precompute shoots RAY_COUNT rays from each TX, records every
# reflection up to MAX_BOUNCES, and builds a spatial hash for fast UAV lookup.
import json, time
import numpy as np
from pathlib import Path
from google.colab import output
from IPython.display import HTML, display
from src.core.scene.domain import Scene, Box, Obstacle, Transmitter, Receiver
from src.core.cache import get_or_compute

MAX_W_2D = 550
W_SIM, L_SIM, Z_SIM = DOMAIN_SIZE[0], DOMAIN_SIZE[1], DOMAIN_SIZE[2]
ASPECT   = L_SIM / W_SIM
CANVAS_W = MAX_W_2D
CANVAS_H = int(MAX_W_2D * ASPECT)
if CANVAS_H > 350:
    CANVAS_H = 350
    CANVAS_W = int(350 / ASPECT)

_HTML_INIT = f"""<script>init({W_SIM},{L_SIM},{Z_SIM},{CANVAS_W},{CANVAS_H});</script>"""

def run_all(obs_data, tx_data, rx_data):
    global scene, static_field
    receiver = Receiver(
        np.array([rx_data["x"], rx_data["y"], rx_data["z"]], dtype=float),
        radius=float(rx_data["eps"]),
    )
    transmitters = [
        Transmitter(
            np.array([t["x"], t["y"], t["z"]], dtype=float),
            float(t["f"]) * 1e9, float(t["p"]), tx_id=t["id"],
        ) for t in tx_data
    ]
    obstacles = [
        Obstacle(
            box_min=[o["x"], o["y"], 0.0],
            box_max=[o["x"]+o["w"], o["y"]+o["l"], o["h"]],
        ) for o in obs_data
    ]
    scene = Scene(
        box=Box(np.zeros(3), np.array(DOMAIN_SIZE, dtype=float)),
        transmitters=transmitters, receiver=receiver, obstacles=obstacles,
    )
    scene.n_rays        = int(RAY_COUNT)
    scene.n_max         = int(MAX_BOUNCES)
    scene.roughness     = float(ROUGHNESS)
    scene.use_physics   = True
    scene.bandwidth_hz  = 20e6
    scene.temperature_c = 20.0

    static_field = get_or_compute(
        scene, seed=42, cell_size=float(SPATIAL_HASH_CELL),
        cache_dir=Path("/content/multipath-passive-radar-sim/cache"),
        verbose=True,
    )
    print(f"Field ready -- {len(static_field.anchors)} anchor paths "
          f"reach the receiver out of {RAY_COUNT:,} rays.")

output.register_callback("run_all", run_all)
_EDITOR_HTML = '\n<script src="https://cdnjs.cloudflare.com/ajax/libs/three.js/r128/three.min.js"></script>\n<script src="https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js"></script>\n<div id="ed" style="background:#000;padding:12px;border-radius:8px;color:#00e5ff;font-family:monospace;border:1px solid #222;max-width:1050px;margin:auto;">\n  <div style="display:flex;justify-content:space-between;font-size:10px;color:#444;margin-bottom:8px;">\n    <span id="dlbl">DOMAIN</span><span id="stag">READY</span>\n  </div>\n  <div style="display:flex;gap:12px;align-items:flex-start;margin-bottom:12px;">\n    <div style="flex:none;" id="cv-wrap">\n      <canvas id="cv" style="border:1px solid #333;background:#000;cursor:crosshair;"></canvas>\n      <div style="display:flex;gap:8px;margin-top:10px;">\n        <button onclick="addObs()" style="background:#111;color:#00e5ff;border:1px solid #00e5ff;cursor:pointer;font-size:9px;padding:4px 10px;">+ BLOCK</button>\n        <button onclick="addTX()"  style="background:#111;color:#ffea00;border:1px solid #ffea00;cursor:pointer;font-size:9px;padding:4px 10px;">+ TX</button>\n        <span style="font-size:8px;color:#444;margin-left:10px;">[DRAG] Move  [SHIFT+CLICK] Delete</span>\n      </div>\n    </div>\n    <div id="t3c" style="flex:1;border:1px solid #222;background:#050505;position:relative;overflow:hidden;"></div>\n  </div>\n  <div id="insp" style="background:#0a0a0a;padding:10px;border:1px solid #1a1a1a;font-size:10px;border-radius:4px;margin-bottom:10px;min-height:45px;">\n    <div id="nsel" style="color:#333;text-align:center;padding-top:10px;">SELECT ELEMENT TO INSPECT</div>\n    <div id="fcon" style="display:none;gap:20px;align-items:center;">\n      <b id="olbl" style="color:#fff;font-size:11px;width:140px;border-right:1px solid #333;"></b>\n      <div id="flds" style="display:flex;gap:15px;flex-grow:1;"></div>\n    </div>\n  </div>\n  <button onclick="doExec()" style="background:#00e5ff;color:#000;padding:10px;border:none;border-radius:2px;cursor:pointer;width:100%;font-weight:bold;font-family:monospace;font-size:11px;letter-spacing:2px;">SYNCHRONIZE &amp; COMPUTE FIELD</button>\n</div>\n<script>\nvar W,L,Z,sX,sY;\nvar rx,txs,obs,si,st;\n\nfunction init(w,l,z,cw,ch){\n  W=w;L=l;Z=z;\n  document.getElementById(\'dlbl\').innerText=\'DOMAIN: \'+W+\' m x \'+L+\' m x \'+Z+\' m\';\n  var cv=document.getElementById(\'cv\');\n  cv.width=cw;cv.height=ch;\n  document.getElementById(\'cv-wrap\').style.width=cw+\'px\';\n  document.getElementById(\'t3c\').style.height=(ch+40)+\'px\';\n  sX=cw/W;sY=ch/L;\n  rx={x:W/2,y:L/2,z:1.5,eps:5.0};\n  txs=[{x:10,y:10,z:Math.min(40,Z),f:2.4,p:10000,id:0}];\n  obs=[{x:50,y:50,w:25,l:25,h:Math.min(40,Z),id:100}];\n  si=-1;st=null;\n  draw();setTimeout(i3,150);\n}\n\nfunction cZ(v){return Math.max(0.1,Math.min(Z,parseFloat(v)||0.1));}\n\nfunction draw(){\n  var cv=document.getElementById(\'cv\'),ctx=cv.getContext(\'2d\');\n  ctx.fillStyle=\'#000\';ctx.fillRect(0,0,cv.width,cv.height);\n  ctx.strokeStyle=\'#111\';ctx.lineWidth=1;\n  for(var i=0;i<=W;i+=50){ctx.beginPath();ctx.moveTo(i*sX,0);ctx.lineTo(i*sX,cv.height);ctx.stroke();}\n  for(var j=0;j<=L;j+=50){ctx.beginPath();ctx.moveTo(0,j*sY);ctx.lineTo(cv.width,j*sY);ctx.stroke();}\n  ctx.fillStyle=\'#ff3d00\';ctx.beginPath();ctx.arc(rx.x*sX,rx.y*sY,6,0,7);ctx.fill();\n  if(st===\'rx\'){ctx.strokeStyle=\'#fff\';ctx.lineWidth=2;ctx.stroke();ctx.lineWidth=1;}\n  ctx.fillStyle=\'#fff\';ctx.font=\'9px monospace\';ctx.fillText(\'RX\',rx.x*sX+8,rx.y*sY+3);\n  txs.forEach(function(t,i){\n    ctx.fillStyle=\'#ffea00\';ctx.beginPath();ctx.arc(t.x*sX,t.y*sY,5,0,7);ctx.fill();\n    if(st===\'tx\'&&si===i){ctx.strokeStyle=\'#fff\';ctx.lineWidth=2;ctx.stroke();ctx.lineWidth=1;}\n    ctx.fillStyle=\'#fff\';ctx.fillText(\'TX\'+t.id,t.x*sX+8,t.y*sY+3);\n  });\n  obs.forEach(function(o,i){\n    ctx.fillStyle=(st===\'obs\'&&si===i)?\'rgba(0,255,255,0.4)\':\'rgba(0,255,255,0.15)\';\n    ctx.fillRect(o.x*sX,o.y*sY,o.w*sX,o.l*sY);\n    ctx.strokeStyle=(st===\'obs\'&&si===i)?\'#fff\':\'#00e5ff\';\n    ctx.strokeRect(o.x*sX,o.y*sY,o.w*sX,o.l*sY);\n  });\n}\n\ndocument.getElementById(\'cv\').onmousedown=function(e){\n  var r=this.getBoundingClientRect();\n  var mx=(e.clientX-r.left)*(this.width/r.width)/sX;\n  var my=(e.clientY-r.top)*(this.height/r.height)/sY;\n  if(Math.hypot(mx-rx.x,my-rx.y)<12){st=\'rx\';si=0;}\n  else{\n    var ti=txs.findIndex(function(t){return Math.hypot(mx-t.x,my-t.y)<10;});\n    if(ti!==-1){st=\'tx\';si=ti;}\n    else{var oi=obs.findIndex(function(o){return mx>=o.x&&mx<=o.x+o.w&&my>=o.y&&my<=o.y+o.l;});st=oi!==-1?\'obs\':null;si=oi;}\n  }\n  if(e.shiftKey&&si!==-1&&st!==\'rx\'){\n    if(st===\'obs\')obs.splice(si,1);else txs.splice(si,1);\n    st=null;s3();\n  }\n  ui();draw();\n};\ndocument.getElementById(\'cv\').onmousemove=function(e){\n  if(si!==-1&&e.buttons===1){\n    var r=this.getBoundingClientRect();\n    var nx=(e.clientX-r.left)*(this.width/r.width)/sX;\n    var ny=(e.clientY-r.top)*(this.height/r.height)/sY;\n    if(st===\'rx\'){rx.x=Math.max(0,Math.min(W,nx));rx.y=Math.max(0,Math.min(L,ny));}\n    else if(st===\'tx\'){txs[si].x=Math.max(0,Math.min(W,nx));txs[si].y=Math.max(0,Math.min(L,ny));}\n    else if(st===\'obs\'){var o=obs[si];o.x=Math.max(0,Math.min(W-o.w,nx-o.w/2));o.y=Math.max(0,Math.min(L-o.l,ny-o.l/2));}\n    draw();ui();s3();\n  }\n};\n\nvar sc3,cam,ren,ctl,rxM,oM={},tM={};\nfunction i3(){\n  var c=document.getElementById(\'t3c\');\n  sc3=new THREE.Scene();sc3.background=new THREE.Color(0x050505);\n  cam=new THREE.PerspectiveCamera(60,c.clientWidth/c.clientHeight,1,2000);\n  cam.position.set(W,Z,L);\n  ren=new THREE.WebGLRenderer({antialias:true});ren.setSize(c.clientWidth,c.clientHeight);\n  c.appendChild(ren.domElement);\n  ctl=new THREE.OrbitControls(cam,ren.domElement);ctl.target.set(W/2,0,L/2);\n  var g=new THREE.GridHelper(Math.max(W,L),20,0x222222,0x111111);g.position.set(W/2,0,L/2);sc3.add(g);\n  sc3.add(new THREE.AmbientLight(0xffffff,0.8));\n  s3();a3();\n}\nfunction s3(){\n  if(!sc3)return;\n  if(!rxM){rxM=new THREE.Mesh(new THREE.SphereGeometry(3),new THREE.MeshLambertMaterial({color:0xff3d00}));sc3.add(rxM);}\n  rxM.position.set(rx.x,rx.z,rx.y);\n  Object.values(tM).forEach(function(m){sc3.remove(m);});tM={};\n  txs.forEach(function(t){var m=new THREE.Mesh(new THREE.SphereGeometry(2),new THREE.MeshLambertMaterial({color:0xffea00}));m.position.set(t.x,t.z,t.y);sc3.add(m);tM[t.id]=m;});\n  Object.values(oM).forEach(function(m){sc3.remove(m);});oM={};\n  obs.forEach(function(o){var m=new THREE.Mesh(new THREE.BoxGeometry(o.w,o.h,o.l),new THREE.MeshLambertMaterial({color:0x00e5ff,transparent:true,opacity:0.3}));m.position.set(o.x+o.w/2,o.h/2,o.y+o.l/2);sc3.add(m);oM[o.id]=m;});\n}\nfunction a3(){requestAnimationFrame(a3);if(ctl)ctl.update();ren.render(sc3,cam);}\n\nfunction ui(){\n  var flds=document.getElementById(\'flds\'),form=document.getElementById(\'fcon\'),none=document.getElementById(\'nsel\');\n  if(!st){form.style.display=\'none\';none.style.display=\'block\';return;}\n  form.style.display=\'flex\';none.style.display=\'none\';\n  if(st===\'rx\'){\n    document.getElementById(\'olbl\').innerText=\'RECEIVER\';\n    flds.innerHTML=\'Z (m): <input type="number" step="0.5" value="\'+rx.z+\'" oninput="rx.z=cZ(this.value);this.value=rx.z;s3()"> Radius (m): <input type="number" step="0.5" value="\'+rx.eps+\'" oninput="rx.eps=parseFloat(this.value)">\';\n  }else if(st===\'tx\'){\n    document.getElementById(\'olbl\').innerText=\'TRANSMITTER TX\'+txs[si].id;\n    flds.innerHTML=\'Z (m): <input type="number" value="\'+txs[si].z+\'" oninput="txs[\'+si+\'].z=cZ(this.value);this.value=txs[\'+si+\'].z;s3()"> Freq (GHz): <input type="number" step="0.1" value="\'+txs[si].f+\'" oninput="txs[\'+si+\'].f=parseFloat(this.value)"> Power (W): <input type="number" value="\'+txs[si].p+\'" oninput="txs[\'+si+\'].p=parseFloat(this.value)">\';\n  }else{\n    document.getElementById(\'olbl\').innerText=\'BUILDING\';\n    flds.innerHTML=\'H (m): <input type="number" value="\'+obs[si].h+\'" oninput="obs[\'+si+\'].h=cZ(this.value);this.value=obs[\'+si+\'].h;s3()"> W (m): <input type="number" value="\'+obs[si].w+\'" oninput="obs[\'+si+\'].w=parseFloat(this.value);draw();s3()"> L (m): <input type="number" value="\'+obs[si].l+\'" oninput="obs[\'+si+\'].l=parseFloat(this.value);draw();s3()">\';\n  }\n}\n\nfunction addObs(){obs.push({x:10,y:10,w:20,l:20,h:Math.min(30,Z),id:Date.now()});draw();s3();}\nfunction addTX(){if(txs.length<4)txs.push({x:20,y:20,z:Math.min(30,Z),f:2.4,p:100,id:txs.length});draw();s3();}\n\nasync function doExec(){\n  document.getElementById(\'stag\').innerText=\'COMPUTING...\';\n  await google.colab.kernel.invokeFunction(\'run_all\',[obs,txs,rx],{});\n  document.getElementById(\'stag\').innerText=\'DONE\';\n}\n</script>\n<style>#flds input{width:65px;background:#000;color:#00e5ff;border:1px solid #333;font-size:11px;padding:3px;font-family:monospace;}</style>\n'
display(HTML(_EDITOR_HTML + _HTML_INIT))


[cache] MISS 06e6226393990659  — running precompute...


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 118 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 118 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 118 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


[cache] SAVE 06e6226393990659  → sf_06e6226393990659.npz  (5.1s)
Field ready -- 24 anchor paths reach the receiver out of 30,000 rays.


# Static Field Visualisation

This section allows you to explore previously computed static fields:
- Browse all cached fields
- Select a specific configuration
- Visualize its multipath structure in 3D


## Workflow

1. **List available fields (Cell 3)**  
   Displays all cached static fields along with key metadata (ray count, anchors, compute time)

2. **Select a field (Cell 4)**  
   Set `SELECTED` to the desired index

3. **Render the scene (Cell 4)**  
   Loads the field and displays an interactive 3D visualization. If no rays reach the receiver, a random subset of MAX_RAYS_3D rays is selected and displayed.


In [4]:
#@title ── 3. Available Cached Fields { display-mode: "form" }
# Lists every static field stored in the cache directory.
# Note the index [i] of the field you want to visualise, then set SELECTED in Cell 4.
import json, time
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import colorsys

_REPO   = Path("/content/multipath-passive-radar-sim")
_CACHE  = _REPO / "cache"
_FIELDS = _CACHE / "precomputed_static_fields"
_REG    = _CACHE / "field_registry.json"

assert _REG.exists(), f"Registry not found: {_REG}\n-> Run Cell 2 first."

registry = json.loads(_REG.read_text())

W = 72
print("=" * W)
print(f"  {'IDX':>3}  {'HASH':16}  {'N_RAYS':>8}  {'ANCHORS':>7}  {'T_PRE':>7}  CREATED")
print("=" * W)
for i, e in enumerate(registry):
    s    = e.get("params_summary", {})
    mark = ">" if i == 0 else " "
    print(f"  {mark} [{i}]  {e['hash']}  "
          f"{s.get('n_rays','?'):>8,}  "
          f"{s.get('anchors','?'):>7}  "
          f"{e.get('precompute_time_s',0):>6.1f}s  "
          f"{e.get('created_at','')[:16]}")
print("=" * W)
print("\n-> Set SELECTED in Cell 4 to the index shown above.")


  IDX  HASH                N_RAYS  ANCHORS    T_PRE  CREATED
  > [0]  06e6226393990659    30,000       24     5.1s  2026-04-01T17:14

-> Set SELECTED in Cell 4 to the index shown above.


In [16]:
#@title ── 4. UAV Interaction Analysis & Field Visualization { display-mode: "form" }
# Kinematic UAV configuration, occlusion/scattering computation, and interactive 3D rendering.

import time
import random
import colorsys
import numpy as np
import plotly.graph_objects as go

# Import simulation core dependencies
from src.core.scene.domain import UAV
from src.core.uav.apply_uav import apply_uav

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION PARAMETERS (UI Panel)
# ─────────────────────────────────────────────────────────────────────────────
SELECTED_FIELD = 0      #@param {type:"number"}
#@markdown
UAV_X          = 94   #@param {type:"slider", min:0, max:200, step:1}
UAV_Y          = 51.0   #@param {type:"slider", min:0, max:200, step:1}
UAV_Z          = 20.0   #@param {type:"slider", min:0, max:100, step:1}
UAV_VX         = 4   #@param {type:"slider", min:0, max:50, step:0.5}
UAV_VY         = 2    #@param {type:"slider", min:0, max:50, step:0.5}
UAV_VZ         = 2    #@param {type:"slider", min:0, max:50, step:0.5}
UAV_RADIUS     = 2.0    #@param {type:"number"}
MAX_RAYS_3D    = 1000   #@param {type:"number"}

# ─────────────────────────────────────────────────────────────────────────────
# 1. STATE VERIFICATION AND DATA RETRIEVAL
# ─────────────────────────────────────────────────────────────────────────────
if 'registry' not in globals() or 'static_field' not in globals() or 'scene' not in globals():
    raise RuntimeError("Dependencies not found. Please execute the pre-computation cells first.")

if not (0 <= SELECTED_FIELD < len(registry)):
    raise ValueError(f"SELECTED_FIELD index ({SELECTED_FIELD}) is out of bounds.")

entry = registry[SELECTED_FIELD]
print(f"Selected Static Field: [{SELECTED_FIELD}] {entry['filename']}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. INTERSECTION EVALUATION AND SCATTERING PATH COMPUTATION
# ─────────────────────────────────────────────────────────────────────────────
# Define UAV state vectors (position and velocity)
pos_vec = np.array([UAV_X, UAV_Y, UAV_Z], dtype=float)
vel_vec = np.array([UAV_VX, UAV_VY, UAV_VZ], dtype=float)

uav_target = UAV(position=pos_vec, velocity=vel_vec, radius=float(UAV_RADIUS))

print(f"Evaluating perturbation for UAV at r = {pos_vec.tolist()} m, v = {vel_vec.tolist()} m/s.")

t_start = time.perf_counter()
vis_rays, occ_rays, uav_rays = apply_uav(static_field, uav_target, scene)
t_elapsed = time.perf_counter() - t_start

print(f"Computation completed in {t_elapsed:.4f} s.")
print(f"  - Preserved intact paths : {len(vis_rays):,}")
print(f"  - Occluded paths (shadow) : {len(occ_rays):,}")
print(f"  - Scattering paths (UAV)  : {len(uav_rays):,}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. VISUALIZATION PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────
# To ensure WebGL stability, uniform sampling is applied if ray counts exceed MAX_RAYS_3D.

def _sample_population(population, quota):
    if len(population) > quota:
        return random.sample(population, quota)
    return population

quota_vis = int(MAX_RAYS_3D * 0.5)
quota_occ = int(MAX_RAYS_3D * 0.25)
quota_uav = int(MAX_RAYS_3D * 0.25)

plot_vis = _sample_population(vis_rays, quota_vis)
plot_occ = _sample_population(occ_rays, quota_occ)
plot_uav = _sample_population(uav_rays, quota_uav)

noise_floor = float(scene.noise_floor_dbm)
tx_ref_pwr = 57.0  # Reference level for thermal gradient normalization

def get_thermal_color(power_dbm: float) -> str:
    """Maps signal power to a thermal color scale."""
    norm_val = float(np.clip((power_dbm - noise_floor) / max(tx_ref_pwr - noise_floor, 1.0), 0.0, 1.0))
    r, g, b = colorsys.hsv_to_rgb(0.67 * (1.0 - norm_val), 1.0, 1.0)
    return f"rgb({int(r*255)}, {int(g*255)}, {int(b*255)})"

# ─────────────────────────────────────────────────────────────────────────────
# 4. PLOTLY OBJECT CONSTRUCTION & UPDATE MENUS
# ─────────────────────────────────────────────────────────────────────────────
fig = go.Figure()

# Logic arrays to control visibility via the dropdown menu
vis_state_static = []  # Visibility for "Precomputed Static Field"
vis_state_applied = [] # Visibility for "UAV Applied Field"

def add_ray_trace(ray, is_occ: bool, is_uav: bool):
    """Appends a ray trace and registers its visibility state."""
    pts = np.array(ray.points)
    if len(pts) < 2: return

    color = get_thermal_color(ray.power_dbm)
    doppler = getattr(ray, 'doppler_shift', 0.0)

    fig.add_trace(go.Scatter3d(
        x=pts[:,0], y=pts[:,1], z=pts[:,2],
        mode="lines",
        line=dict(color=color, width=2),
        opacity=0.6,
        showlegend=False,
        hovertemplate=f"Power: {ray.power_dbm:.2f} dBm<br>Doppler: {doppler:.2f} Hz<extra></extra>"
    ))

    if is_occ:
        vis_state_static.append(True)
        vis_state_applied.append(False)
    elif is_uav:
        vis_state_static.append(False)
        vis_state_applied.append(True)
    else: # is_vis (Intact)
        vis_state_static.append(True)
        vis_state_applied.append(True)

# Sequential processing of ray collections
for r in plot_vis: add_ray_trace(r, is_occ=False, is_uav=False)
for r in plot_occ: add_ray_trace(r, is_occ=True,  is_uav=False)
for r in plot_uav: add_ray_trace(r, is_occ=False, is_uav=True)

# ── Domain Geometry Integration ──────────────────────────────────────────────

def add_static_geometry_trace(trace_obj):
    """Adds invariant geometric elements to both visibility menus."""
    fig.add_trace(trace_obj)
    vis_state_static.append(True)
    vis_state_applied.append(True)

# Obstacle mesh generation (Buildings)
for ob in scene.obstacles:
    mn, mx = ob.box_min, ob.box_max
    xs, ys, zs = [], [], []
    C = [[mn[0],mn[1],mn[2]], [mx[0],mn[1],mn[2]], [mx[0],mx[1],mn[2]], [mn[0],mx[1],mn[2]],
         [mn[0],mn[1],mx[2]], [mx[0],mn[1],mx[2]], [mx[0],mx[1],mx[2]], [mn[0],mx[1],mx[2]]]
    for a,b in [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]:
        xs += [C[a][0], C[b][0], None]; ys += [C[a][1], C[b][1], None]; zs += [C[a][2], C[b][2], None]

    add_static_geometry_trace(go.Scatter3d(
        x=xs, y=ys, z=zs, mode="lines",
        line=dict(color="rgba(0, 229, 255, 0.15)", width=1),
        showlegend=False, hoverinfo="skip"
    ))

# Transmitters (TX) and Receiver (RX)
tx_pos = np.array([tx.position for tx in scene.transmitters])
add_static_geometry_trace(go.Scatter3d(
    x=tx_pos[:,0], y=tx_pos[:,1], z=tx_pos[:,2], mode="markers+text",
    text=[f"TX{tx.tx_id}" for tx in scene.transmitters], textposition="bottom center",
    textfont=dict(color="#FF4444", size=10), marker=dict(color="#FF4444", size=6), name="TX"
))

rx_pos = scene.receiver.position
add_static_geometry_trace(go.Scatter3d(
    x=[rx_pos[0]], y=[rx_pos[1]], z=[rx_pos[2]], mode="markers+text",
    text=["RX"], textposition="bottom center",
    textfont=dict(color="#00FFFF", size=10), marker=dict(color="#00FFFF", size=8), name="RX"
))

# UAV Topological Marker (Exclusive to "Applied" view)
fig.add_trace(go.Scatter3d(
    x=[uav_target.position[0]], y=[uav_target.position[1]], z=[uav_target.position[2]],
    mode="markers+text", text=["UAV"], textposition="top center",
    textfont=dict(color="#ffea00", size=12),
    marker=dict(color="#ffea00", size=8, symbol="diamond", line=dict(color="white", width=1.5)),
    name="UAV"
))
vis_state_static.append(False)
vis_state_applied.append(True)

# ── Graphical Interface and Menu Logic ──────────────────────────────────────
BG_COLOR = "#0a0a12"
ax_params = dict(showgrid=False, zeroline=False, showticklabels=False, backgroundcolor=BG_COLOR, title="")

fig.update_layout(
    paper_bgcolor=BG_COLOR, plot_bgcolor=BG_COLOR, font=dict(color="white", size=11),
    height=800, margin=dict(l=0, r=0, t=60, b=0),
    scene=dict(
        bgcolor=BG_COLOR, xaxis=ax_params, yaxis=ax_params, zaxis=ax_params, aspectmode="data",
        camera=dict(eye=dict(x=1.3, y=1.3, z=0.8))
    ),
    updatemenus=[dict(
        type="dropdown", direction="down",
        x=0.02, y=0.98, xanchor="left", yanchor="top",
        bgcolor="#1a1a24", bordercolor="#333", font=dict(color="#fff"),
        buttons=list([
            dict(
                label="Precomputed Static Field",
                method="update",
                args=[{"visible": vis_state_static},
                      {"title.text": f"Precomputed Static Field (Sample: {len(plot_vis) + len(plot_occ)} rays)"}]
            ),
            dict(
                label="UAV Applied Field",
                method="update",
                args=[{"visible": vis_state_applied},
                      {"title.text": f"UAV Applied Field (Sample: {len(plot_vis) + len(plot_uav)} rays, Doppler Evaluated)"}]
            )
        ])
    )],
    title=dict(
        text=f"Precomputed Static Field (Sample: {len(plot_vis) + len(plot_occ)} rays)",
        x=0.5, y=0.95, xanchor="center", font=dict(size=14, color="#eeeeee")
    ),
    showlegend=False
)

# Initialize figure in the static state
fig.update_traces(visible=False)
for idx, is_visible in enumerate(vis_state_static):
    fig.data[idx].visible = is_visible

fig.show()

Selected Static Field: [0] sf_06e6226393990659.npz
Evaluating perturbation for UAV at r = [94.0, 51.0, 20.0] m, v = [4.0, 2.0, 2.0] m/s.
Computation completed in 0.0140 s.
  - Preserved intact paths : 24
  - Occluded paths (shadow) : 0
  - Scattering paths (UAV)  : 1
